# Single-slice DLPFC

In [ ]:
from pathlib import Path
import sys
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import scanpy as sc
import torch
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score
import matplotlib.pyplot as plt
cwd = Path.cwd().resolve()
PROJECT_ROOT = next(
    (path for path in (cwd, *cwd.parents) if (path / "SpaDiff").is_dir()),
    cwd,
)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import SpaDiff as sd
from SpaDiff.spatial import spatial_reconstruction
from SpaDiff.utils import mclust_R, set_seed
from SpaDiff.topology import build_simplicial_operators, to_torch_operators

## Configuration

In [ ]:
SEED = 42
SAMPLE_ID = "151674"
N_CLUSTERS = 7
N_NEIGHBORS = 10
TRAINING_EPOCHS = 500


MAX_ORDER = 2
SIMPLEX_ORDERS = (
    (0,) if MAX_ORDER == 0
    else tuple(range(1, MAX_ORDER + 1))
)

DSM_WEIGHTING = "variance"

DSM_LOSS_WEIGHT = 1.0
BATCH_LOSS_WEIGHT = 0.0  
BATCH_POSTERIOR_SCALE = 0.0  
PRIOR_KL_LOSS_WEIGHT = 1.0  

DATA_ROOT = Path("path/DLPFC")
print("DATA_ROOT =", DATA_ROOT)

set_seed(SEED)
torch.backends.cudnn.deterministic = True
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print("device =", device)
print(f"loss weights: DSM={DSM_LOSS_WEIGHT}, batch={BATCH_LOSS_WEIGHT}, prior_KL={PRIOR_KL_LOSS_WEIGHT}")

## Data loading

In [ ]:
sample_dir = DATA_ROOT / SAMPLE_ID
adata = sc.read_visium(sample_dir)
adata.var_names_make_unique()

adata.layers["counts"] = adata.X.copy()
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)
sc.pp.highly_variable_genes(adata, flavor="seurat_v3", layer="counts", n_top_genes=3000, subset=True)
truth = pd.read_csv(sample_dir / "truth.txt", sep="\t", header=None, index_col=0)
truth.columns = ["Truth"]
adata.obs["Truth"] = truth.reindex(adata.obs_names)["Truth"]
adata

## Simplicial complex

In [ ]:
adata, adjacency = spatial_reconstruction(adata, alpha=2.0, n_neighbors=N_NEIGHBORS)

operators = to_torch_operators(build_simplicial_operators(adjacency, max_order=MAX_ORDER), device=device)

sc.tl.pca(adata, n_comps=50)
features = torch.as_tensor(np.asarray(adata.obsm["X_pca"]), dtype=torch.float32, device=device)

In [ ]:
print("features:", tuple(features.shape))
print("operator nnz by order:", {order: operator._nnz() for order, operator in operators.items()})


## Conditional VP-SDE training

In [ ]:
config = sd.SpaDiffConfig(
    data_dim=features.shape[1],
    condition_input_dim=features.shape[1],
    propagation_steps=5,
    propagation_alpha=0.4,
    dropout=0.1,
    topology_projection_dropout=0.0,
    learnable_propagation=False,
    topology_residual=True,
    topology_output_normalization="feature",
    simplex_orders=SIMPLEX_ORDERS,
    dsm_weighting=DSM_WEIGHTING,
    dsm_weight=DSM_LOSS_WEIGHT,
    batch_alignment_weight=BATCH_LOSS_WEIGHT,
    batch_posterior_weight=BATCH_POSTERIOR_SCALE,
    prior_kl_weight=PRIOR_KL_LOSS_WEIGHT,
)
model = sd.SpaDiff(config).to(device)
adata = model.fit_transform(
    adata,
    features,
    operators,
    batch_key=None,
    epochs=TRAINING_EPOCHS,
    progress=True,
    ode_steps=200,
)

## Spatial domains and ARI

In [ ]:
labels = mclust_R(adata, num_cluster=N_CLUSTERS, used_obsm="spadiff",pca_num=20, random_seed=SEED)
adata.obs["mclust"] = pd.Categorical(labels.astype(str))
valid = adata.obs[["mclust", "Truth"]].dropna()
ari = round(adjusted_rand_score(valid["Truth"], valid["mclust"]),3)
nmi = normalized_mutual_info_score(valid["Truth"], valid["mclust"])

print(f"ARI = {ari:.3f}")
print(f"NMI = {nmi:.3f}")

palette = ["#6D1A9C", "#D1D1D1", "#F56867", "#59BE86", "#FEB915", "#C798EE", "#7495D3"]
sc.pl.spatial(
    adata, img_key="hires", color="mclust", palette=palette,
    title=f"SpaDiff | ARI={ari:.3f}", legend_loc=None,
    frameon=False, spot_size=120,
    show=False,
)
# plt.savefig("../result/spadiff_"+SAMPLE_ID+"_"+str(ari)+".pdf", bbox_inches='tight')

In [ ]:
# output_file ="../result/spadiff_"+SAMPLE_ID+"_"+str(ari)+".h5ad"  # Compressed output path

# adata.write_h5ad(output_file, compression="gzip")